# 04 — Image Classification with CNNs and Transfer Learning

**Goal:** classify 37 cat and dog breeds from photographs, compare a small CNN trained from scratch with ImageNet transfer learning, explain predictions, and export a deployment-safe TorchScript model.

[Open in Google Colab](https://colab.research.google.com/github/Jorgoluka100/uni_projects/blob/main/04_Image_Classification_with_CNNs_and_Transfer_Learning.ipynb)

**Portfolio evidence:** real public images, fixed train/validation/test boundaries, reproducible training, macro-F1 and top-3 accuracy, error analysis, confidence-based review, Grad-CAM, model card, and automated acceptance tests. Reported results are produced by this notebook run—not typed claims.

Dataset: [Oxford-IIIT Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/) (7,349 images; 37 breeds). The dataset is for non-commercial research use; check the source terms before other use.

## 1. Setup and reproducibility

In [1]:
# Colab already includes most packages. Uncomment only if an import fails.
# !pip -q install torch torchvision scikit-learn seaborn

import copy, json, os, random, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, top_k_accuracy_score
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT=Path('/content/pet_project' if Path('/content').exists() else './pet_project')
ROOT.mkdir(parents=True,exist_ok=True)
print({'torch':torch.__version__,'device':str(DEVICE),'seed':SEED})

{'torch': '2.8.0+cpu', 'device': 'cpu', 'seed': 42}


## 2. Load and audit the real image data

The official `trainval` split is divided reproducibly into training and validation sets. The untouched official test split is used once for final evaluation.

In [2]:
raw_train=OxfordIIITPet(ROOT,split='trainval',target_types='category',download=True)
raw_test=OxfordIIITPet(ROOT,split='test',target_types='category',download=True)
classes=raw_train.classes; n_classes=len(classes)

# Stratified 80/20 split inside the official trainval partition.
targets=np.asarray(raw_train._labels)-1
rng=np.random.default_rng(SEED); train_idx=[]; val_idx=[]
for y in range(n_classes):
    ids=np.flatnonzero(targets==y); rng.shuffle(ids); cut=int(.8*len(ids))
    train_idx.extend(ids[:cut]); val_idx.extend(ids[cut:])
train_idx=np.array(sorted(train_idx)); val_idx=np.array(sorted(val_idx))
test_idx=np.arange(len(raw_test))
assert set(train_idx).isdisjoint(val_idx)
print({'all_images':len(raw_train)+len(raw_test),'train':len(train_idx),'validation':len(val_idx),'test':len(test_idx),'classes':n_classes})

# Decode audit; fail loudly rather than silently training on corrupt files.
bad=[]
for ds_name,ds in [('trainval',raw_train),('test',raw_test)]:
    for i,p in enumerate(ds._images):
        try:
            with Image.open(p) as im: im.verify()
        except Exception as e: bad.append((ds_name,i,str(e)))
print('corrupt_images:',len(bad)); assert not bad

counts=pd.Series(targets).value_counts().sort_index()
fig,ax=plt.subplots(figsize=(12,4)); ax.bar(classes,counts); ax.tick_params(axis='x',rotation=90); ax.set(title='Official trainval images per breed',ylabel='images'); plt.tight_layout(); plt.show()

{'all_images': 7349, 'train': 2861, 'validation': 719, 'test': 3669, 'classes': 37}
corrupt_images: 0


## 3. Image pipeline and leakage controls

In [3]:
IMAGE_SIZE=160
weights=MobileNet_V3_Small_Weights.DEFAULT
mean,std=weights.transforms().mean,weights.transforms().std
train_tf=transforms.Compose([transforms.Resize((176,176)),transforms.RandomResizedCrop(IMAGE_SIZE,scale=(.75,1.0)),transforms.RandomHorizontalFlip(),transforms.ColorJitter(.15,.15,.15,.05),transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_tf=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),transforms.ToTensor(),transforms.Normalize(mean,std)])

class PetView(Dataset):
    def __init__(self,base,indices,tf): self.base,self.indices,self.tf=base,np.asarray(indices),tf
    def __len__(self): return len(self.indices)
    def __getitem__(self,j):
        i=int(self.indices[j]); image,target=self.base[i]
        return self.tf(image),target,i

train_ds=PetView(raw_train,train_idx,train_tf); val_ds=PetView(raw_train,val_idx,eval_tf); test_ds=PetView(raw_test,test_idx,eval_tf)
BATCH=64 if DEVICE.type=='cuda' else 32
loaders={k:DataLoader(v,batch_size=BATCH,shuffle=(k=='train'),num_workers=2 if DEVICE.type=='cuda' else 0,pin_memory=DEVICE.type=='cuda') for k,v in {'train':train_ds,'val':val_ds,'test':test_ds}.items()}
x,y,_=next(iter(loaders['train'])); print(x.shape,y.min().item(),y.max().item())
fig,axs=plt.subplots(2,4,figsize=(11,6)); denorm=lambda z:(z*torch.tensor(std)[:,None,None]+torch.tensor(mean)[:,None,None]).clamp(0,1)
for ax,img,label in zip(axs.flat,x[:8],y[:8]): ax.imshow(denorm(img).permute(1,2,0)); ax.set_title(classes[label]); ax.axis('off')
plt.tight_layout(); plt.show()

torch.Size([32, 3, 160, 160]) 1 36


## 4. Baseline: a CNN trained from scratch

In [4]:
class SmallCNN(nn.Module):
    def __init__(self,n=n_classes):
        super().__init__(); self.features=nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(128,192,3,padding=1),nn.BatchNorm2d(192),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
        self.classifier=nn.Sequential(nn.Flatten(),nn.Dropout(.25),nn.Linear(192,n))
    def forward(self,x): return self.classifier(self.features(x))

def run_epoch(model,loader,optimizer=None):
    train=optimizer is not None; model.train(train); total=0.; ys=[]; ps=[]
    for xb,yb,_ in loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        if train: optimizer.zero_grad(set_to_none=True)
        out=model(xb); loss=nn.functional.cross_entropy(out,yb,label_smoothing=.05 if train else 0)
        if train: loss.backward(); optimizer.step()
        total+=loss.item()*len(yb); ys.extend(yb.cpu().tolist()); ps.extend(out.argmax(1).detach().cpu().tolist())
    return total/len(loader.dataset),accuracy_score(ys,ps),f1_score(ys,ps,average='macro')

EPOCHS_SCRATCH=3 if DEVICE.type=='cuda' else 1
scratch=SmallCNN().to(DEVICE); opt=torch.optim.AdamW(scratch.parameters(),lr=2e-3,weight_decay=1e-4)
scratch_history=[]
for e in range(EPOCHS_SCRATCH):
    tr=run_epoch(scratch,loaders['train'],opt); va=run_epoch(scratch,loaders['val'])
    scratch_history.append({'epoch':e+1,'train_loss':tr[0],'train_acc':tr[1],'val_loss':va[0],'val_acc':va[1],'val_macro_f1':va[2]}); print(scratch_history[-1])
scratch_val_f1=scratch_history[-1]['val_macro_f1']

{'epoch': 1, 'train_loss': 3.5633528794245635, 'train_acc': 0.04928346731911919, 'val_loss': 3.5143242021595156, 'val_acc': 0.059805285118219746, 'val_macro_f1': 0.029287370332826152}


## 5. Transfer learning: ImageNet MobileNetV3

First train only a new classification head. On a GPU, then unfreeze the final feature blocks for one low-learning-rate fine-tuning epoch.

In [5]:
transfer=mobilenet_v3_small(weights=weights)
for p in transfer.features.parameters(): p.requires_grad=False
transfer.classifier[3]=nn.Linear(transfer.classifier[3].in_features,n_classes)
transfer=transfer.to(DEVICE)
HEAD_EPOCHS=5 if DEVICE.type=='cuda' else 2
opt=torch.optim.AdamW(transfer.classifier.parameters(),lr=2e-3,weight_decay=1e-4)
history=[]; best_state=None; best_f1=-1
for e in range(HEAD_EPOCHS):
    tr=run_epoch(transfer,loaders['train'],opt); va=run_epoch(transfer,loaders['val'])
    row={'stage':'head','epoch':e+1,'train_loss':tr[0],'train_acc':tr[1],'val_loss':va[0],'val_acc':va[1],'val_macro_f1':va[2]}; history.append(row); print(row)
    if va[2]>best_f1: best_f1=va[2]; best_state=copy.deepcopy(transfer.state_dict())
if DEVICE.type=='cuda':
    transfer.load_state_dict(best_state)
    for p in transfer.features[-3:].parameters(): p.requires_grad=True
    opt=torch.optim.AdamW(filter(lambda p:p.requires_grad,transfer.parameters()),lr=1e-4,weight_decay=1e-4)
    tr=run_epoch(transfer,loaders['train'],opt); va=run_epoch(transfer,loaders['val'])
    row={'stage':'fine_tune','epoch':1,'train_loss':tr[0],'train_acc':tr[1],'val_loss':va[0],'val_acc':va[1],'val_macro_f1':va[2]}; history.append(row); print(row)
    if va[2]>best_f1: best_f1=va[2]; best_state=copy.deepcopy(transfer.state_dict())
transfer.load_state_dict(best_state); transfer.eval()

{'stage': 'head', 'epoch': 1, 'train_loss': 2.225576041628955, 'train_acc': 0.41698706745893044, 'val_loss': 1.2042694789395048, 'val_acc': 0.6286509040333796, 'val_macro_f1': 0.618814546565412}
{'stage': 'head', 'epoch': 2, 'train_loss': 1.5195243271111358, 'train_acc': 0.6298497029010836, 'val_loss': 1.2316058108554595, 'val_acc': 0.6161335187760779, 'val_macro_f1': 0.6130520034666005}


## 6. Untouched test evaluation

In [6]:
@torch.inference_mode()
def predict(model,loader):
    logits=[]; ys=[]; ids=[]
    model.eval()
    for xb,yb,ib in loader:
        logits.append(model(xb.to(DEVICE)).cpu()); ys.append(yb); ids.append(ib)
    logits=torch.cat(logits); return torch.cat(ys).numpy(),logits.softmax(1).numpy(),torch.cat(ids).numpy()

y_test,prob_test,id_test=predict(transfer,loaders['test']); pred_test=prob_test.argmax(1)
metrics={'test_accuracy':accuracy_score(y_test,pred_test),'test_macro_f1':f1_score(y_test,pred_test,average='macro'),'test_top3_accuracy':top_k_accuracy_score(y_test,prob_test,k=3,labels=np.arange(n_classes))}
print(json.dumps(metrics,indent=2))
report=pd.DataFrame(classification_report(y_test,pred_test,target_names=classes,output_dict=True,zero_division=0)).T
display(report.sort_values('f1-score').head(10))
cm=confusion_matrix(y_test,pred_test)
fig,ax=plt.subplots(figsize=(13,11)); sns.heatmap(cm,cmap='Blues',xticklabels=classes,yticklabels=classes,ax=ax); ax.set(xlabel='Predicted',ylabel='Actual',title='Test confusion matrix'); plt.tight_layout(); plt.show()

{
  "test_accuracy": 0.5854456255110384,
  "test_macro_f1": 0.5630680161722362,
  "test_top3_accuracy": 0.8310166257835923
}
                            precision    recall  f1-score  support
Abyssinian                   0.000000  0.000000  0.000000     98.0
American Pit Bull Terrier    0.376623  0.290000  0.327684    100.0
Staffordshire Bull Terrier   0.538462  0.235955  0.328125     89.0
Great Pyrenees               1.000000  0.210000  0.347107    100.0
Basset Hound                 0.866667  0.260000  0.400000    100.0
Ragdoll                      0.652174  0.300000  0.410959    100.0
American Bulldog             0.680851  0.320000  0.435374    100.0
British Shorthair            0.560606  0.370000  0.445783    100.0
Chihuahua                    0.452830  0.480000  0.466019    100.0
English Cocker Spaniel       0.561644  0.410000  0.473988    100.0


## 7. Confidence guardrail and error analysis

Low confidence is routed to human review. The threshold is selected on validation data, never on the test labels.

In [7]:
y_val,prob_val,_=predict(transfer,loaders['val']); conf_val=prob_val.max(1); ok_val=prob_val.argmax(1)==y_val
candidates=np.linspace(.25,.9,66); viable=[]
for t in candidates:
    keep=conf_val>=t
    if keep.mean()>=.50: viable.append((ok_val[keep].mean(),keep.mean(),t))
review_threshold=max(viable)[2] if viable else .5
conf=prob_test.max(1); keep=conf>=review_threshold
guardrail={'threshold_from_validation':review_threshold,'test_coverage':keep.mean(),'test_accuracy_when_accepted':(pred_test[keep]==y_test[keep]).mean(),'test_review_rate':1-keep.mean()}
print(json.dumps(guardrail,indent=2))

errors=pd.DataFrame({'id':id_test,'actual':[classes[i] for i in y_test],'predicted':[classes[i] for i in pred_test],'confidence':conf})
errors=errors[y_test!=pred_test].sort_values('confidence',ascending=False)
display(errors.head(12))
fig,axs=plt.subplots(2,4,figsize=(12,7))
for ax,(_,r) in zip(axs.flat,errors.head(8).iterrows()):
    ax.imshow(raw_test[int(r.id)][0]); ax.set_title(f"A: {r.actual}\nP: {r.predicted} ({r.confidence:.2f})",fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()

{
  "threshold_from_validation": 0.62,
  "test_coverage": 0.49114200054510765,
  "test_accuracy_when_accepted": 0.8085460599334073,
  "test_review_rate": 0.5088579994548923
}
        id                      actual           predicted  confidence
2387  2387                  Pomeranian             Samoyed    0.982249
1035  1035                   Chihuahua  Miniature Pinscher    0.978956
1226  1226      English Cocker Spaniel        Newfoundland    0.974837
368    368                Basset Hound              Beagle    0.974251
3457  3457  Staffordshire Bull Terrier           Shiba Inu    0.966763
2409  2409                  Pomeranian             Samoyed    0.966039
2308  2308                     Persian             Samoyed    0.959767
2985  2985            Scottish Terrier             Samoyed    0.958238
843    843                       Boxer                 Pug    0.957862
624    624                      Birman             Siamese    0.953438
645    645                      Birman      

## 8. Grad-CAM explanation, with hooks removed safely

In [8]:
class GradCAM:
    def __init__(self,model,layer):
        self.model=model; self.activations=None; self.gradients=None
        self.handles=[layer.register_forward_hook(self._forward)]
    def _forward(self,m,i,o):
        self.activations=o.detach()
        if o.requires_grad: o.register_hook(lambda g:setattr(self,'gradients',g.detach()))
    def __call__(self,x,target):
        x=x.detach().requires_grad_(True); self.model.zero_grad(set_to_none=True); score=self.model(x)[0,target]; score.backward()
        w=self.gradients.mean((2,3),keepdim=True); cam=(w*self.activations).sum(1).relu()[0]
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8); return cam.cpu().numpy()
    def close(self):
        for h in self.handles: h.remove()
        self.handles=[]

img,y,_=test_ds[0]; cam_engine=GradCAM(transfer,transfer.features[-1]); predicted=int(transfer(img[None].to(DEVICE)).argmax(1)); cam=cam_engine(img[None].to(DEVICE),predicted); cam_engine.close()
cam_big=np.array(Image.fromarray((cam*255).astype('uint8')).resize((IMAGE_SIZE,IMAGE_SIZE)))/255
display_img=denorm(img).permute(1,2,0).numpy()
fig,axs=plt.subplots(1,2,figsize=(8,4)); axs[0].imshow(display_img); axs[0].set_title(f'Actual: {classes[y]}'); axs[1].imshow(display_img); axs[1].imshow(cam_big,cmap='jet',alpha=.42); axs[1].set_title(f'Grad-CAM: {classes[predicted]}'); [a.axis('off') for a in axs]; plt.tight_layout(); plt.show()

## 9. Hook-free TorchScript export

The previous `Modules that have backward hooks assigned can't be compiled` failure is prevented by rebuilding a clean model from the saved state dictionary and asserting that no hooks remain before tracing.

In [9]:
def hook_count(model):
    return sum(len(m._forward_hooks)+len(m._forward_pre_hooks)+len(m._backward_hooks) for m in model.modules())

export_model=mobilenet_v3_small(weights=None)
export_model.classifier[3]=nn.Linear(export_model.classifier[3].in_features,n_classes)
export_model.load_state_dict({k:v.detach().cpu() for k,v in transfer.state_dict().items()}); export_model.eval()
assert hook_count(export_model)==0, 'Export model must be hook-free'
example=torch.randn(1,3,IMAGE_SIZE,IMAGE_SIZE)
with torch.inference_mode():
    eager=export_model(example); traced=torch.jit.trace(export_model,example); scripted=traced(example)
max_delta=(eager-scripted).abs().max().item(); assert max_delta<1e-4
export_path=ROOT/'pet_breed_mobilenet_v3_small.ts'; traced.save(str(export_path))
print({'torchscript_path':str(export_path),'hook_count':hook_count(export_model),'max_output_delta':max_delta,'size_mb':export_path.stat().st_size/1e6})

{'torchscript_path': 'pet_project/pet_breed_mobilenet_v3_small.ts', 'hook_count': 0, 'max_output_delta': 0.0, 'size_mb': 6.635964}


## 10. Acceptance tests, model card, and CV evidence

In [10]:
checks={
 'real_dataset_complete':len(raw_train)+len(raw_test)==7349,
 '37_classes':n_classes==37,
 'train_validation_disjoint':set(train_idx).isdisjoint(val_idx),
 'no_corrupt_images':len(bad)==0,
 'probabilities_sum_to_one':np.allclose(prob_test.sum(1),1,atol=1e-5),
 'finite_metrics':all(np.isfinite(list(metrics.values()))),
 'hook_free_export':hook_count(export_model)==0,
 'torchscript_matches_eager':max_delta<1e-4,
 'artifact_exists':export_path.exists()
}
display(pd.Series(checks,name='passed').to_frame()); assert all(checks.values())

summary={**metrics,**guardrail,'scratch_validation_macro_f1':scratch_val_f1,'transfer_validation_macro_f1':best_f1}
print('RUN-DERIVED SUMMARY'); print(json.dumps(summary,indent=2))
print(f"CV bullet: Built a PyTorch pet-breed classifier on 7,349 real images across 37 classes; transfer learning achieved {metrics['test_macro_f1']:.3f} macro-F1 and {metrics['test_top3_accuracy']:.1%} top-3 accuracy, with Grad-CAM, confidence-based review and verified hook-free TorchScript export.")

model_card={
 'model':'MobileNetV3-Small transfer learning','task':'37-class Oxford-IIIT pet breed classification','intended_use':'portfolio demonstration and assisted categorisation only','not_for':'identity, safety-critical, veterinary or unrestricted commercial use','data':'official Oxford-IIIT Pet trainval/test; stratified validation split from trainval','metrics':metrics,'guardrail':guardrail,'limitations':['breed labels can be visually ambiguous','dataset may not represent mixed breeds or real deployment conditions','confidence is not a guarantee of correctness','licence and privacy review required before deployment']}
print(json.dumps(model_card,indent=2))

                           passed
real_dataset_complete        True
37_classes                   True
train_validation_disjoint    True
no_corrupt_images            True
probabilities_sum_to_one     True
finite_metrics               True
hook_free_export             True
torchscript_matches_eager    True
artifact_exists              True
RUN-DERIVED SUMMARY
{
  "test_accuracy": 0.5854456255110384,
  "test_macro_f1": 0.5630680161722362,
  "test_top3_accuracy": 0.8310166257835923,
  "threshold_from_validation": 0.62,
  "test_coverage": 0.49114200054510765,
  "test_accuracy_when_accepted": 0.8085460599334073,
  "test_review_rate": 0.5088579994548923,
  "scratch_validation_macro_f1": 0.029287370332826152,
  "transfer_validation_macro_f1": 0.618814546565412
}
CV bullet: Built a PyTorch pet-breed classifier on 7,349 real images across 37 classes; transfer learning achieved 0.563 macro-F1 and 83.1% top-3 accuracy, with Grad-CAM, confidence-based review and verified hook-free TorchScript expor

## Conclusion

Transfer learning is the production candidate because ImageNet features provide a much stronger starting point than the small scratch CNN. The test set remains untouched until final evaluation, and confidence-based review reduces automation risk. Grad-CAM is useful for debugging attention patterns, not proof of causal reasoning. Before deployment, validate on target-domain images, monitor class and confidence drift, review licensing, and define a human escalation process.